# UDS NRC v3 QLoRA Smoke Test

순서대로 실행한다. 전체 학습 전에 1.5B base baseline과 50-step smoke만 확인한다.

In [ ]:
!nvidia-smi
!git clone https://github.com/LSC18/uds-nrc-finetuning-dataset.git
%cd uds-nrc-finetuning-dataset

In [ ]:
# Colab의 CUDA 호환 PyTorch는 유지한다. 이 실험에 필요 없는 vision/audio 패키지는 충돌 방지를 위해 제거한다.
!python -m pip uninstall -y -q torchvision torchaudio torchtext
!python -m pip install -q -r requirements-colab.txt
!python -c 'import torch, transformers, peft, bitsandbytes; print("torch", torch.__version__, "cuda", torch.version.cuda, "available", torch.cuda.is_available()); print("transformers", transformers.__version__)'

In [ ]:
!python scripts/validate_dataset.py --data-dir full_v3
!shasum -a 256 -c SHA256SUMS
!python scripts/preflight.py --config configs/qlora_v3_smoke_1.5b.json --report reports/preflight_v3_1.5b_gpu.json

## Base model baseline
먼저 held-out test 20개로 generation 경로를 확인한다.

In [ ]:
!python evaluate_exact_match.py --model-name Qwen/Qwen2.5-1.5B-Instruct --data-dir full_v3 --limit 20 --output reports/v3_base_1.5b_first20.jsonl

## 50-step QLoRA smoke
adapter와 final_metrics.json이 생성되고 loss가 NaN이 아니면 성공이다.

In [ ]:
!python train_qlora.py --config configs/qlora_v3_smoke_1.5b.json

In [ ]:
!python evaluate_exact_match.py --adapter-path outputs/v3-smoke-1.5b --data-dir full_v3 --output reports/v3_smoke_1.5b.jsonl
!cat outputs/v3-smoke-1.5b/final_metrics.json

## 다음 단계
smoke가 성공한 경우에만 `configs/qlora_v3_full_1.5b.json`으로 전체 학습한다. 출력 폴더와 reports를 먼저 Google Drive에 복사한다.